# broker

> Many conversations, one copy of the weights.

A local model is expensive to load and expensive to hold in memory, so a second process wanting one is usually told to wait. `ChatBroker` serves a Unix socket instead: every client gets its own conversation, and they share the one engine the broker built.

The isolation is real. A client sends strings and gets a reply back, and cannot register a tool, read another client's history, or reach the engine directly.

::: {.callout-note}
This module is a candidate for extraction. It is process plumbing that happens to carry chats, and shares nothing with the rest of urai but `Resp`.
:::

In [ ]:
#| default_exp broker

In [ ]:
#| export
import json, os, socket, threading
from fastcore.all import Path
from urai.core import Resp

In [ ]:
#| hide
import tempfile, time
from fastcore.test import test_eq, test_fail

## The wire

Length-prefixed JSON, because a socket gives you a byte stream and not messages. The cap on an incoming frame is there so a client cannot ask the broker to allocate a gigabyte before it has said anything.

In [ ]:
#| export
MAX_FRAME = 1_000_000   #: refuse a request larger than this, before allocating for it

def _send_json(sock, obj):
    data = json.dumps(obj).encode()
    sock.sendall(len(data).to_bytes(4, 'big') + data)

def _recv_json(sock):
    def read(n):
        out = bytearray()
        while len(out) < n:
            if not (part := sock.recv(n - len(out))): raise EOFError
            out.extend(part)
        return bytes(out)
    n = int.from_bytes(read(4), 'big')
    if n > MAX_FRAME: raise ValueError('broker request too large')
    return json.loads(read(n))

In [ ]:
a, b = socket.socketpair()
_send_json(a, {'op': 'send', 'msg': 'hello'})
test_eq(_recv_json(b), {'op': 'send', 'msg': 'hello'})

In [ ]:
a.sendall((MAX_FRAME + 1).to_bytes(4, 'big'))
test_fail(lambda: _recv_json(b), contains='too large')
a.close()
test_fail(lambda: _recv_json(b), contains='')     # a closed peer is EOF, not a hang
b.close()

## Checking what a client sent

Everything crossing the socket is checked before it reaches the engine. A client is another process, so its requests are input rather than arguments.

In [ ]:
#| export
def _brok_msgs(req):
    "An `open` request's `messages`: a list of strings, checked before the engine sees it."
    msgs = req.get('messages') or []
    if not isinstance(msgs, list) or not all(isinstance(m, str) for m in msgs):
        raise ValueError('invalid conversation')
    return msgs

def _brok_limit(req):
    "A `send` request's `max_output_tokens`: a positive int, or nothing."
    n = req.get('max_output_tokens')
    if n is None: return None
    if isinstance(n, bool) or not isinstance(n, int) or n < 1: raise ValueError('invalid token limit')
    return n

In [ ]:
test_eq(_brok_msgs({'messages': ['a', 'b']}), ['a', 'b'])
test_eq(_brok_msgs({}), [])
test_fail(lambda: _brok_msgs({'messages': [{'role': 'user'}]}), contains='invalid conversation')
test_fail(lambda: _brok_msgs({'messages': 'not a list'}), contains='invalid conversation')

In [ ]:
test_eq(_brok_limit({'max_output_tokens': 100}), 100)
test_eq(_brok_limit({}), None)
test_fail(lambda: _brok_limit({'max_output_tokens': 0}), contains='invalid token limit')
test_fail(lambda: _brok_limit({'max_output_tokens': True}), contains='invalid token limit')
test_fail(lambda: _brok_limit({'max_output_tokens': '100'}), contains='invalid token limit')

## The broker

One lock guards the engine, so turns from different clients queue rather than interleave. A second guards the broker's own bookkeeping, so a client connecting while another is disconnecting does not race.

The socket is created `0600`. It drives a model on its owner's behalf, and anyone who can write to it can spend the owner's compute.

In [ ]:
#| export
class ChatBroker:
    "Serve isolated text chats over a Unix socket, all sharing one local engine."
    def __init__(self, address, chat_cls, engine=None, model_id=None, **engine_kw):
        self.address, self.chat_cls, self.engine = str(address), chat_cls, engine
        self.model_id, self.engine_kw, self._own_engine = model_id, engine_kw, engine is None
        self.lock, self.state_lock = threading.Lock(), threading.Lock()
        self.listener, self.thread = None, None
        self.clients, self.handlers, self.stopping, self._bound = set(), set(), False, False

    def _mk_engine(self):
        "The shared engine from the backend's own `create_engine`."
        mid = {'model_id': self.model_id} if self.model_id is not None else {}
        return self.chat_cls.create_engine(**mid, **self.engine_kw)

    def _shut_engine(self):
        "Release the engine, but only one this broker built itself."
        if self._own_engine and (c := getattr(self.engine, 'close', None)): c(); self.engine = None

    def _free_socket(self):
        "Clear a socket file a dead broker left behind. A live one is never taken over."
        path = Path(self.address)
        if not path.exists(): return
        if not path.is_socket(): raise RuntimeError(f'broker path exists and is not a socket: {self.address}')
        probe = socket.socket(socket.AF_UNIX)
        try: probe.connect(self.address)
        except ConnectionRefusedError: path.unlink()
        else: raise RuntimeError(f'broker already running at {self.address}')
        finally: probe.close()

    def start(self):
        "Build the engine if this broker owns one, then serve `address` until `close`."
        with self.state_lock:
            if self.listener is not None: return self
            self._free_socket(); listener = None
            try:
                if self.engine is None: self.engine = self._mk_engine()
                listener = socket.socket(socket.AF_UNIX)
                listener.bind(self.address)
                os.chmod(self.address, 0o600)   # this socket drives a model: its owner, nobody else
                listener.listen()
            except Exception:
                if listener is not None: listener.close()
                self._shut_engine(); Path(self.address).unlink(missing_ok=True)
                raise
            self.stopping, self.listener, self._bound = False, listener, True
            self.thread = threading.Thread(target=self._serve, args=(listener,), daemon=True)
            self.thread.start()
        return self

    def _serve(self, listener):
        "Accept clients until the listener closes, one handler thread each."
        while True:
            try: conn, _ = listener.accept()
            except OSError: break
            handler = threading.Thread(target=self._handle, args=(conn,), daemon=True)
            with self.state_lock:
                if self.stopping: conn.close(); break
                self.clients.add(conn); self.handlers.add(handler)
            handler.start()

    def _handle(self, conn):
        "One client: `open` its chat, run `send` turns through it, drop it on `close`."
        chat = None
        try:
            while not self.stopping:
                req = _recv_json(conn)
                if not isinstance(req, dict) or (op := req.get('op')) not in {'open', 'send', 'close'}:
                    raise ValueError('invalid broker request')
                if op == 'open':
                    if chat is not None: raise ValueError('conversation already open')
                    msgs = _brok_msgs(req)
                    with self.lock:
                        chat = self.chat_cls(engine=self.engine, messages=msgs, tools=())
                    _send_json(conn, {'ok': True})
                elif op == 'send':
                    if chat is None or not isinstance(req.get('msg'), str):
                        raise ValueError('invalid message')
                    limit = _brok_limit(req)
                    with self.lock: res = chat(req['msg'], max_output_tokens=limit)
                    _send_json(conn, {'response': res})
                else: _send_json(conn, {'ok': True}); break
        except (EOFError, OSError): pass
        except Exception as e:
            try: _send_json(conn, {'error': f'{type(e).__name__}: {e}'})
            except OSError: pass
        finally:
            if chat is not None:
                with self.lock: chat.close()
            conn.close()
            with self.state_lock:
                self.clients.discard(conn); self.handlers.discard(threading.current_thread())

    def _wake(self):
        """Connect once to wake `accept`, which closing the listener alone may not wake."""
        s = socket.socket(socket.AF_UNIX)
        try: s.connect(self.address)
        except OSError: pass
        finally: s.close()

    @staticmethod
    def _drop(conn):
        "Close a client connection and wake its handler. `close` alone does not wake a blocked `recv`."
        for f in (lambda: conn.shutdown(socket.SHUT_RDWR), conn.close):
            try: f()
            except OSError: pass

    def close(self, timeout=5):
        "Stop serving, drop every client, and release the engine this broker built."
        with self.state_lock:
            listener, thread, handlers = self.listener, self.thread, list(self.handlers)
            clients, bound = list(self.clients), self._bound
            self.stopping, self.listener, self.thread, self._bound = True, None, None, False
        if listener is not None: self._wake(); listener.close()
        for conn in clients: self._drop(conn)
        for t in [thread, *handlers]:
            # a handler wedged inside a turn must not hold shutdown up for ever
            if t is not None and t is not threading.current_thread(): t.join(timeout)
        with self.lock: self._shut_engine()
        if bound: Path(self.address).unlink(missing_ok=True)   # never a socket it did not bind

    def __enter__(self): return self.start()
    def __exit__(self, *a): self.close()

## The client

In [ ]:
#| export
class BrokerChat:
    "One isolated conversation on a `ChatBroker`: strings out, a `Resp` back."
    def __init__(self, address, messages=None):
        messages = list(messages or [])
        if not all(isinstance(m, str) for m in messages):
            raise TypeError('broker messages must be strings')
        self.conn, self.lock = socket.socket(socket.AF_UNIX), threading.Lock()
        try:
            self.conn.connect(str(address))
            self._call({'op': 'open', 'messages': messages})
        except Exception: self.conn.close(); raise

    def _call(self, req):
        _send_json(self.conn, req); res = _recv_json(self.conn)
        if 'error' in res: raise RuntimeError(res['error'])
        return res

    def __call__(self, msg, max_output_tokens=None):
        "Send one turn through the shared engine."
        if not isinstance(msg, str): raise TypeError('broker messages must be strings')
        with self.lock:
            return Resp(self._call({'op': 'send', 'msg': msg,
                                    'max_output_tokens': max_output_tokens})['response'])

    def close(self):
        "Let the broker drop this conversation. Idempotent."
        with self.lock:
            if getattr(self, 'conn', None) is None: return
            try: self._call({'op': 'close'})
            except (OSError, EOFError): pass
            self.conn.close(); self.conn = None

    def __enter__(self): return self
    def __exit__(self, *a): self.close()

## Trying it

`_EngineChat` stands in for a real backend: it takes an engine, keeps its own history, and echoes. That is enough to show that two clients get separate conversations off one engine.

In [ ]:
class _Engine:
    "A stand-in for a loaded model. Counts how many were built and whether it was released."
    built, closed = 0, False
    def __init__(self): type(self).built += 1
    def close(self): type(self).closed = True

class _EngineChat:
    "The smallest thing `ChatBroker` will serve."
    def __init__(self, engine=None, messages=None, tools=(), **kw):
        self.engine, self.hist = engine, list(messages or [])
    @classmethod
    def create_engine(cls, **kw): return _Engine()
    def __call__(self, msg, max_output_tokens=None):
        self.hist.append(msg)
        return {'role': 'assistant', 'content': f'[{len(self.hist)}] {msg}'}
    def close(self): pass

sock = str(Path(tempfile.mkdtemp())/'broker.sock')

In [ ]:
with ChatBroker(sock, _EngineChat) as b:
    test_eq(_Engine.built, 1)
    test_eq(oct(os.stat(sock).st_mode)[-3:], '600')
    with BrokerChat(sock) as c1, BrokerChat(sock) as c2:
        test_eq(c1('hello')['content'], '[1] hello')
        test_eq(c1('again')['content'], '[2] again')
        test_eq(c2('hello')['content'], '[1] hello')    # its own conversation, from scratch
test_eq(_Engine.built, 1)                               # ...off the one engine
test_eq(_Engine.closed, True)                           # released on close
test_eq(Path(sock).exists(), False)                     # ...and the socket file with it

In [ ]:
with ChatBroker(sock, _EngineChat) as b:
    with BrokerChat(sock, messages=['earlier turn']) as c:
        test_eq(c('now')['content'], '[2] now')         # the history it opened with counts
    test_fail(lambda: BrokerChat(sock, messages=[{'role': 'user'}]), contains='must be strings')
    with BrokerChat(sock) as c:
        test_fail(lambda: c(123), contains='must be strings')

In [ ]:
# a bad request is an error on that client's connection, not a broker that falls over
with ChatBroker(sock, _EngineChat) as b:
    c = BrokerChat(sock)
    test_fail(lambda: c._call({'op': 'nonsense'}), contains='invalid broker request')
    c.close()
    test_eq(c.close(), None)                            # idempotent
    with BrokerChat(sock) as ok: test_eq(ok('still here')['content'], '[1] still here')

In [ ]:
# a live broker is never taken over
with ChatBroker(sock, _EngineChat) as b:
    test_fail(lambda: ChatBroker(sock, _EngineChat).start(), contains='already running')

# a non-socket collision is never deleted
Path(sock).touch()
test_fail(lambda: ChatBroker(sock, _EngineChat).start(), contains='not a socket')
test_eq(Path(sock).is_file(), True)
Path(sock).unlink()

# a crashed broker leaves an unreachable socket inode, which the next broker replaces
stale = socket.socket(socket.AF_UNIX)
stale.bind(sock); stale.close()
test_eq(Path(sock).is_socket(), True)
with ChatBroker(sock, _EngineChat) as b:
    with BrokerChat(sock) as c: test_eq(c('after a crash')['content'], '[1] after a crash')

In [ ]:
# shutting down is prompt, even with the accept loop blocked and a client still connected
import time
b = ChatBroker(sock, _EngineChat).start()
c = BrokerChat(sock)
t = time.time(); b.close()
assert time.time() - t < 2
test_eq(b.thread, None)

In [ ]:
# an engine handed in is the caller's to release, not the broker's
_Engine.closed = False
e = _Engine()
with ChatBroker(sock, _EngineChat, engine=e) as b:
    with BrokerChat(sock) as c: test_eq(c('borrowed')['content'], '[1] borrowed')
test_eq(_Engine.closed, False)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()